# Predict Model — Employee Performance
**IABAC Certified Data Scientist — Project Code: 10281**

Demonstrates both trained models using the actual persisted preprocessing pipeline (fitted
encoders, not just their printed mapping), so a **raw** record — with real category strings like
`EmpDepartment='Sales'` — can be scored directly, the way HR would actually use this in practice.


In [1]:
import pandas as pd
import numpy as np
import joblib
import json

label_encoders = joblib.load('../../models/label_encoders.pkl')
binary_maps = joblib.load('../../models/binary_maps.pkl')

hiring_model = joblib.load('../../models/hiring_model.pkl')
hiring_scaler = joblib.load('../../models/hiring_scaler.pkl')
hiring_features = joblib.load('../../models/hiring_feature_names.pkl')

diagnostic_model = joblib.load('../../models/diagnostic_model.pkl')
diagnostic_scaler = joblib.load('../../models/diagnostic_scaler.pkl')
diagnostic_features = joblib.load('../../models/diagnostic_feature_names.pkl')

with open('../../models/model_metadata.json') as f:
    meta = json.load(f)

print('Hiring model:', meta['hiring_model']['model_name'], '| test accuracy:', round(meta['hiring_model']['test_accuracy'],4), '| macro-F1:', round(meta['hiring_model']['macro_f1'],4))
print('Diagnostic model:', meta['diagnostic_model']['model_name'], '| test accuracy:', round(meta['diagnostic_model']['test_accuracy'],4), '| macro-F1:', round(meta['diagnostic_model']['macro_f1'],4))

Hiring model: XGBoost | test accuracy: 0.4167 | macro-F1: 0.3309
Diagnostic model: Random Forest | test accuracy: 0.9417 | macro-F1: 0.901


## 1. A Reusable, Raw-Input Encoding Function

This is the piece the first draft of this project was missing: a function that takes a raw record (real strings, not pre-encoded integers) and applies the *exact same* fitted encoders used during training.

In [2]:
def encode_raw_record(raw: dict) -> dict:
    """Transforms a raw record (real category strings) into the encoded values the models
    expect, using the encoders persisted by data_processing.ipynb."""
    encoded = dict(raw)
    for col, mapping in binary_maps.items():
        if col in encoded:
            encoded[col] = mapping[encoded[col]]
    for col, le in label_encoders.items():
        if col in encoded:
            encoded[col] = int(le.transform([encoded[col]])[0])
    return encoded

# Quick self-check: round-trip a known category through its encoder
example = {'EmpDepartment': 'Sales', 'Gender': 'Male'}
print('Raw:', example)
print('Encoded:', encode_raw_record(example))

Raw: {'EmpDepartment': 'Sales', 'Gender': 'Male'}
Encoded: {'EmpDepartment': 5, 'Gender': 1}


## 2. Hiring Model — Score a New Candidate

Only fields a recruiter would actually have on an applicant: demographics, education, prior work history, and the role/department/level being hired into. No satisfaction scores, no tenure, no salary-hike history — because none of that exists yet for someone who hasn't been hired.

In [3]:
def predict_for_candidate(raw_candidate: dict):
    encoded = encode_raw_record(raw_candidate)
    row = pd.DataFrame([encoded])[hiring_features]
    X_in = hiring_scaler.transform(row) if meta['hiring_model']['needs_scaling'] else row
    pred = hiring_model.predict(X_in)[0] + meta['hiring_model']['label_offset']
    proba = None
    if hasattr(hiring_model, 'predict_proba'):
        p = hiring_model.predict_proba(X_in)[0]
        classes = sorted(set(pd.read_csv('../../data/processed/employee_performance_processed.csv')['PerformanceRating']))
        proba = dict(zip(classes, p.round(3)))
    return pred, proba

candidate = {
    'Age': 29,
    'Gender': 'Female',
    'EducationBackground': 'Marketing',
    'MaritalStatus': 'Single',
    'EmpDepartment': 'Sales',
    'EmpJobRole': 'Sales Executive',
    'EmpJobLevel': 2,
    'BusinessTravelFrequency': 'Travel_Rarely',
    'DistanceFromHome': 8,
    'EmpEducationLevel': 3,
    'NumCompaniesWorked': 2,
    'TotalWorkExperienceInYears': 6,
}

pred_rating, proba = predict_for_candidate(candidate)
print('Candidate profile:', candidate)
print()
print('Predicted PerformanceRating:', pred_rating)
print('Class probabilities:', proba)

Candidate profile: {'Age': 29, 'Gender': 'Female', 'EducationBackground': 'Marketing', 'MaritalStatus': 'Single', 'EmpDepartment': 'Sales', 'EmpJobRole': 'Sales Executive', 'EmpJobLevel': 2, 'BusinessTravelFrequency': 'Travel_Rarely', 'DistanceFromHome': 8, 'EmpEducationLevel': 3, 'NumCompaniesWorked': 2, 'TotalWorkExperienceInYears': 6}

Predicted PerformanceRating: 2
Class probabilities: {2: np.float32(0.462), 3: np.float32(0.302), 4: np.float32(0.237)}


**How to read this:** given the Hiring Model's modest macro-F1 (0.331, see `train_model.ipynb`
Section 1), this prediction should support a hiring conversation, not replace it — e.g. as one
input alongside interview performance and reference checks, not a pass/fail gate. This is
consistent with the CEO's own stated concern in the project brief about not making high-stakes
calls off a single automated signal.

## 3. Diagnostic Model — Review an Existing Employee

Uses the full feature set (including tenure, satisfaction, compensation growth) — valid here because this is about a person who already has an employment history at INX, not a candidate.

In [4]:
df_enc = pd.read_csv('../../data/processed/employee_performance_processed.csv')
from sklearn.model_selection import train_test_split
X_diag_full = df_enc[diagnostic_features]
y_full = df_enc['PerformanceRating']
_, X_test_diag, _, y_test_diag = train_test_split(X_diag_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

X_in = diagnostic_scaler.transform(X_test_diag) if meta['diagnostic_model']['needs_scaling'] else X_test_diag
preds = diagnostic_model.predict(X_in) + meta['diagnostic_model']['label_offset']

from sklearn.metrics import accuracy_score
print('Reproduced diagnostic test accuracy:', round(accuracy_score(y_test_diag, preds), 4),
      '(training-time value was', round(meta['diagnostic_model']['test_accuracy'], 4), ')')

sample_out = X_test_diag.copy()
sample_out['Actual_Rating'] = y_test_diag.values
sample_out['Predicted_Rating'] = preds
sample_out.head(10)

Reproduced diagnostic test accuracy: 0.9417 (training-time value was 0.9417 )


,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,...,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,Actual_Rating,Predicted_Rating
811,35,0,3,1,1,0,2,23,4,3,...,4,3,3,2,2,2,2,0,3,3
1149,26,1,5,2,1,3,2,24,3,3,...,1,3,1,1,0,0,0,1,3,3
662,36,1,2,1,5,13,2,17,2,3,...,12,1,1,4,2,1,3,0,2,3
542,53,1,3,1,2,4,2,24,4,2,...,11,2,3,4,3,1,2,0,2,2
858,34,0,1,0,1,0,2,6,4,3,...,12,3,3,1,0,0,0,0,3,3
191,38,1,4,1,4,16,1,3,4,3,...,8,2,3,2,2,2,2,0,3,3
479,29,1,3,2,4,9,2,3,1,2,...,10,3,3,10,9,1,5,0,2,2
1,47,1,2,2,5,13,2,14,4,4,...,20,2,3,7,7,1,7,0,3,3
659,44,1,1,1,4,5,2,1,4,2,...,10,5,3,2,0,2,2,0,3,3
1136,41,1,1,2,5,13,0,10,2,4,...,16,3,3,14,3,1,10,0,3,3


## Summary
- `encode_raw_record()` demonstrates the fitted encoders (persisted in `data_processing.ipynb`)
  correctly transforming **raw** category strings — the gap flagged in review is now closed.
- The **Hiring Model** scores a new candidate using only pre-employment-knowable fields, with an
  explicit reminder of its honest, modest accuracy (see `train_model.ipynb` for the full
  majority-class-baseline comparison) so it's used as a decision-support signal, not an
  automated gate.
- The **Diagnostic Model** reproduces its training-time accuracy exactly when reloaded, and is
  used here to review an existing employee's predicted vs. actual rating — the correct use case
  for a model trained on tenure/satisfaction/compensation-history features.